# 04 — Tension Analysis

This notebook quantifies the **Hubble tension** — the statistical
discrepancy between early-universe (CMB) and late-universe (SNe Ia)
measurements of $H_0$.

We compute:
1. The Gaussian tension metric $T = \Delta H_0 / \sigma_\Delta$
2. The Bayesian probability $P(H_0 \leq 67.4 | \mathcal{D})$
3. A comparison whisker plot of all major $H_0$ measurements

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from mcmc import load_chain, tension_metric, p_planck
from plots import plot_h0_comparison

plt.rcParams.update({'font.size': 12})
print('Modules loaded.')

## 4.1 Load MCMC Results

In [ ]:
chain = load_chain('../mcmc_chain.npy')
N_BURNIN = 500; THIN = 15
flat_samples = chain[N_BURNIN::THIN, :, :].reshape(-1, 2)
H0_samples   = flat_samples[:, 0]
Om_samples   = flat_samples[:, 1]

H0_q = np.percentile(H0_samples, [16, 50, 84])
Om_q = np.percentile(Om_samples,  [16, 50, 84])

H0_med = H0_q[1]; H0_lo = H0_q[1]-H0_q[0]; H0_hi = H0_q[2]-H0_q[1]
Om_med = Om_q[1]; Om_lo = Om_q[1]-Om_q[0]; Om_hi = Om_q[2]-Om_q[1]

print(f'H0 = {H0_med:.2f} +{H0_hi:.2f} / -{H0_lo:.2f} km/s/Mpc')
print(f'Om = {Om_med:.3f} +{Om_hi:.3f} / -{Om_lo:.3f}')

## 4.2 Gaussian Tension Metric

The Gaussian tension statistic is:

$$T = \frac{|H_0^{\rm this\,work} - H_0^{\rm Planck}|}{\sqrt{\sigma_{\rm SH0ES}^2 + \sigma_{\rm Planck}^2}}$$

We use the full SH0ES uncertainty $\sigma = 1.04$ km/s/Mpc in the
denominator (not the tighter SNe-only uncertainty), as this represents
the complete distance ladder error budget and is the standard convention
in the literature.

In [ ]:
H0_planck     = 67.4
sigma_planck  = 0.5
sigma_shoes   = 1.04   # full SH0ES 2022 error budget
H0_mle        = 73.01  # from notebook 02
sigma_mle     = 0.37

# Tension from MCMC
T_mcmc = tension_metric(H0_med, sigma_shoes, H0_planck, sigma_planck)

# Tension from MLE (for comparison)
T_mle  = abs(H0_mle - H0_planck) / np.sqrt(sigma_shoes**2 + sigma_planck**2)

# SNe-only tension (for reference — not the standard metric)
T_sne  = abs(H0_med - H0_planck) / np.sqrt(H0_hi**2 + sigma_planck**2)

print('=' * 55)
print('  TENSION METRICS')
print('=' * 55)
print(f'  Delta H0                     = {abs(H0_med - H0_planck):.2f} km/s/Mpc')
print(f'  sigma_Delta (full SH0ES)     = {np.sqrt(sigma_shoes**2+sigma_planck**2):.2f}')
print(f'  T (MCMC, full SH0ES budget)  = {T_mcmc:.2f} sigma')
print(f'  T (MLE,  full SH0ES budget)  = {T_mle:.2f} sigma')
print(f'  T (SNe-only, for reference)  = {T_sne:.2f} sigma')
print('=' * 55)
print('  Note: SNe-only tension excludes Cepheid calibration')
print('  systematics and should NOT be compared to published tensions.')

## 4.3 Bayesian Probability $P(H_0 \leq 67.4 | \mathcal{D})$

The fraction of MCMC samples at or below the Planck central value
gives a direct Bayesian measure of incompatibility.

In [ ]:
pp = p_planck(H0_samples, H0_planck)
print(f'P(H0 <= {H0_planck} | data) = {pp:.6f}')
if pp == 0.0:
    print(f'  i.e. 0 out of {len(H0_samples):,} samples fall at or below Planck value')
    print('  This confirms the posterior has negligible overlap with H0=67.4')
else:
    print(f'  i.e. 1 in {1/pp:.0f} samples')

In [ ]:
# Visualise the posterior with Planck value marked
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(H0_samples, bins=80, density=True, color='steelblue',
        edgecolor='white', linewidth=0.3, alpha=0.85,
        label='MCMC posterior samples')
ax.axvline(H0_planck, color='teal', linestyle='--', linewidth=2,
           label=f'Planck 2018: $H_0$ = {H0_planck}')
ax.axvline(H0_med, color='crimson', linestyle='-', linewidth=2,
           label=f'This work: $H_0$ = {H0_med:.2f}')
ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 2],
                 H0_samples.min(), H0_planck,
                 color='teal', alpha=0.08,
                 label=f'$P(H_0 \\leq {H0_planck}) = {pp:.4f}$')

ax.set_xlabel(r'$H_0$ (km/s/Mpc)', fontsize=13)
ax.set_ylabel('Probability Density', fontsize=13)
ax.set_title(r'MCMC Posterior for $H_0$ with Planck Reference',
             fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle=':', alpha=0.4)
plt.tight_layout()
plt.savefig('../figures/fig_tension_posterior.pdf', bbox_inches='tight')
plt.show()

## 4.4 H₀ Comparison Whisker Plot

In [ ]:
plot_h0_comparison(
    H0_this_work_mcmc=H0_med,
    H0_this_work_mle=H0_mle,
    sigma_mcmc_lo=H0_lo,
    sigma_mcmc_hi=H0_hi,
    sigma_mle=sigma_mle
)

## 4.5 Final Results Summary

| Method | $H_0$ (km/s/Mpc) | Tension vs Planck |
|--------|------------------|-------------------|
| Frequentist MLE | 73.01 ± 0.37 | ~4.9σ |
| Bayesian MCMC | 73.04 +0.37/−0.36 | ~4.9σ |
| Planck 2018 | 67.4 ± 0.5 | — |
| SH0ES 2022 | 73.04 ± 1.04 | ~5σ |

Both methods agree to within 0.04 km/s/Mpc, confirming the
implementation is correct and the ~4.9σ Hubble tension is robust.

---
*Analysis complete. All figures saved to `figures/`.*